# 8 - Feature Contribution Analysis (SHAP)

**Covers:** Section 4.9.3(B), reported as a **co-primary result**, not as supporting evidence.

TreeExplainer on the trained XGBoost Alignment-Augmented model yields exact Shapley values for tree ensembles, so the reported contribution carries no sampling error of its own. Values are aggregated into three groups -- **the alignment score**, **the twelve audio descriptors**, and **the VADER lyric sentiment** -- and the alignment score's rank among all individual features is reported.

Not repeated for Random Forest: Section 4.2.2 limits RF's role to the R2/RMSE/MAE robustness check.

Section 4.9.3(B): *"Where the difference in R2 yields a single aggregate number, SHAP shows **where** the alignment feature contributes... and remains informative even where that aggregate difference is small."* That is why this notebook matters most if notebook 7 returned a null.

**Produces:** `artifacts/shap_group_contributions.csv`, `artifacts/shap_feature_ranks.csv`, `artifacts/shap_where_it_contributes.csv`, `artifacts/fig_shap_*.png`.

In [ ]:
import os
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

sys.path.insert(0, os.path.abspath('.'))
import modeling_config as mc

pd.set_option('display.width', 160)
PLOT_SAMPLE = 20_000   # plots only; group means use the full test set

df = mc.load_corpus()
splits = pd.read_csv(mc.artifact('splits.csv'))
df = df.merge(splits[['id', 'partition']], on='id', how='inner')
df = df[df['partition'].isin(['train', 'val', 'test'])].reset_index(drop=True)

X_all, blocks = mc.build_feature_frame(df)
mc.assert_clean(X_all)
cols = mc.columns_for('alignment_augmented', blocks)
is_test = (df['partition'] == 'test').to_numpy()
X_test = X_all.loc[is_test, cols]
meta = df.loc[is_test, ['id', 'genre', 'popularity']].reset_index(drop=True)

model = joblib.load(mc.artifact('models', 'xgboost__alignment_augmented.joblib'))
print('test rows:', X_test.shape[0], ' features:', X_test.shape[1])

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap_df = pd.DataFrame(shap_values, columns=cols)
print('shap matrix:', shap_df.shape)
print('expected value (base prediction): %.4f' % float(explainer.expected_value))

# local accuracy: contributions plus the base value reconstruct the prediction
recon = shap_df.sum(axis=1).to_numpy() + float(explainer.expected_value)
actual = model.predict(X_test)
print('max |reconstruction - prediction|: %.3e' % np.abs(recon - actual).max())

## Group-level contributions

Mean absolute SHAP value per feature group. The audio group is the twelve descriptors, with the twelve `key_*` dummies counted together as the single `key` descriptor they encode.

In [ ]:
group_of = {}
for c in cols:
    if c in mc.ALIGNMENT_COLS:
        group_of[c] = 'alignment score'
    elif c in mc.LYRIC_COLS:
        group_of[c] = 'lyric sentiment (VADER)'
    else:
        group_of[c] = 'twelve audio descriptors'

mean_abs = shap_df.abs().mean()
groups = mean_abs.groupby(pd.Series(group_of)).sum().sort_values(ascending=False)
group_table = pd.DataFrame({
    'mean_abs_shap': groups,
    'share_pct': 100 * groups / groups.sum(),
    'n_columns': pd.Series(group_of).value_counts(),
})
print(group_table.round(4).to_string())
group_table.to_csv(mc.artifact('shap_group_contributions.csv'))

print('\nSection 4.9.3(B) reports this table plus the alignment score\'s rank below.')

In [ ]:
# individual-feature ranking, with the key dummies collapsed back to one descriptor
collapsed = mean_abs.copy()
key_cols = [c for c in cols if c.startswith('key_')]
if key_cols:
    collapsed = collapsed.drop(index=key_cols)
    collapsed['key (12 one-hot levels)'] = mean_abs[key_cols].sum()

ranks = (collapsed.sort_values(ascending=False)
         .rename('mean_abs_shap').reset_index()
         .rename(columns={'index': 'feature'}))
ranks['rank'] = np.arange(1, len(ranks) + 1)
print(ranks.round(4).to_string(index=False))

alignment_rank = int(ranks.loc[ranks['feature'] == 'alignment_gap', 'rank'].iloc[0])
print('\nalignment_gap ranks %d of %d individual features.' % (alignment_rank, len(ranks)))
ranks.to_csv(mc.artifact('shap_feature_ranks.csv'), index=False)

## Where the alignment feature contributes

The part of Section 4.9.3(B) that survives a small aggregate ΔR2: which tracks, and which regions of the popularity range.

In [ ]:
where = meta.copy()
where['shap_alignment'] = shap_df['alignment_gap'].to_numpy()
where['abs_shap_alignment'] = where['shap_alignment'].abs()
where['alignment_gap'] = X_test['alignment_gap'].to_numpy()
where['band'] = pd.cut(where['popularity'], bins=[0, 20, 40, 60, 80, 100],
                       labels=['1-20', '21-40', '41-60', '61-80', '81-100'])

print('=== mean |SHAP| for alignment_gap, by popularity band ===')
print(where.groupby('band', observed=True)
      .agg(n=('id', 'size'), mean_abs_shap=('abs_shap_alignment', 'mean'),
           mean_signed_shap=('shap_alignment', 'mean')).round(4).to_string())

print('\n=== mean |SHAP| for alignment_gap, by genre ===')
print(where.groupby('genre')
      .agg(n=('id', 'size'), mean_abs_shap=('abs_shap_alignment', 'mean'),
           mean_signed_shap=('shap_alignment', 'mean'))
      .sort_values('mean_abs_shap', ascending=False).round(4).to_string())

where.to_csv(mc.artifact('shap_where_it_contributes.csv'), index=False)

In [ ]:
rng = np.random.default_rng(mc.RANDOM_SEED)
take = rng.choice(len(X_test), size=min(PLOT_SAMPLE, len(X_test)), replace=False)

shap.summary_plot(shap_values[take], X_test.iloc[take], max_display=18, show=False)
plt.tight_layout()
plt.savefig(mc.artifact('fig_shap_beeswarm.png'), dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(where['alignment_gap'].to_numpy()[take],
           where['shap_alignment'].to_numpy()[take], s=3, alpha=0.25)
ax.axhline(0, color='k', lw=1, ls='--')
ax.axvline(0, color='k', lw=1, ls='--')
ax.set_xlabel('alignment_gap')
ax.set_ylabel('SHAP value for alignment_gap (popularity points)')
ax.set_title('What the model does with the alignment gap')
fig.tight_layout()
fig.savefig(mc.artifact('fig_shap_dependence.png'), dpi=150)
plt.show()

---
### Reading the dependence plot

This is the figure that answers the research question qualitatively. A monotone rising or falling band means the model learned a directional effect -- lyrics brighter than the music pushes predicted popularity one way consistently. A U or inverted-U means **magnitude** of mismatch matters regardless of direction, which would be a finding about emotional complexity rather than emotional positivity. A flat, structureless cloud means the model found nothing usable, which is the visual form of a null result.

Section 4.9.3(B) also notes this analysis directly addresses the limitation Zhang et al. (2026) identify: independent audio and lyric importance rankings cannot express whether the two signals reinforce or contradict one another. A single feature encoding exactly that relationship can.

Next: `9_robustness.ipynb`.